# Emotions in Music Analysis
**Kaden Van Atta | Spring 2024**  
**Tools:** Python, Pandas, NumPy, Plotly  
**Dataset:** 2,000 Spotify tracks across 10 genres

---

### Overview
This analysis explores the emotional landscape of music by examining Spotify audio features — **valence**, **energy**, **tempo**, **danceability**, **acousticness**, and **instrumentalness** across ten distinct genres. By mapping these features to emotional quadrants and visualizing patterns interactively, the project reveals how genre shapes emotional experience in music.

**Key Questions:**
- Which genres occupy distinct emotional territories?
- How do valence and energy interact to define emotional tone?
- What audio features differentiate "happy" music from "intense" music?
- Are there clear emotional clusters across genres?


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load dataset
df = pd.read_csv('data/spotify_data.csv')

print(f"Dataset: {df.shape[0]} tracks across {df['genre'].nunique()} genres")
print(f"Genres: {', '.join(sorted(df['genre'].unique()))}")
print(f"\nEmotion distribution:")
print(df['emotion'].value_counts())
df.describe().round(3)


---
## 1. Valence vs. Energy — The Emotional Map
Valence measures musical positivity (0 = dark/sad, 1 = happy/euphoric). Energy captures intensity and activity. Together they form four emotional quadrants that map to distinct moods.

In [ ]:
# Quadrant scatter: valence vs energy, colored by genre
fig = px.scatter(
    df, x='valence', y='energy', color='genre',
    hover_data=['danceability', 'tempo', 'acousticness'],
    opacity=0.6,
    title='Valence vs. Energy by Genre — The Emotional Map',
    labels={'valence': 'Valence (Negativity → Positivity)', 'energy': 'Energy (Calm → Intense)'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    width=900, height=600
)

# Quadrant dividers
fig.add_hline(y=0.5, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=0.5, line_dash='dash', line_color='gray', opacity=0.5)

# Quadrant labels
annotations = [
    dict(x=0.75, y=0.92, text="😤 Angry / Intense", showarrow=False, font=dict(size=11, color='gray'), xref='x', yref='y'),
    dict(x=0.75, y=0.08, text="😊 Content / Peaceful", showarrow=False, font=dict(size=11, color='gray'), xref='x', yref='y'),
    dict(x=0.18, y=0.92, text="😠 Dark / Intense", showarrow=False, font=dict(size=11, color='gray'), xref='x', yref='y'),
    dict(x=0.18, y=0.08, text="😢 Sad / Melancholic", showarrow=False, font=dict(size=11, color='gray'), xref='x', yref='y'),
]
fig.update_layout(annotations=annotations, plot_bgcolor='white', paper_bgcolor='white',
                  legend_title='Genre', font=dict(family='Arial'))
fig.update_xaxes(showgrid=True, gridcolor='#f0f0f0', range=[0, 1])
fig.update_yaxes(showgrid=True, gridcolor='#f0f0f0', range=[0, 1])
fig.show()


---
## 2. Emotion Distribution by Genre
How do genres distribute across the four emotional quadrants? This reveals each genre's emotional personality.

In [ ]:
# Stacked bar: emotion distribution per genre
emotion_dist = df.groupby(['genre', 'emotion']).size().reset_index(name='count')
emotion_dist['pct'] = emotion_dist.groupby('genre')['count'].transform(lambda x: x / x.sum() * 100)

emotion_colors = {
    'Happy / Excited': '#f59e0b',
    'Content / Peaceful': '#10b981',
    'Angry / Intense': '#ef4444',
    'Sad / Dark': '#6366f1'
}

fig2 = px.bar(
    emotion_dist, x='genre', y='pct', color='emotion',
    color_discrete_map=emotion_colors,
    title='Emotion Distribution by Genre (%)',
    labels={'pct': 'Percentage of Tracks (%)', 'genre': 'Genre', 'emotion': 'Emotional Tone'},
    width=900, height=520
)
fig2.update_layout(
    barmode='stack', plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial'), legend_title='Emotional Tone'
)
fig2.update_xaxes(showgrid=False)
fig2.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig2.show()


---
## 3. Average Audio Features Radar Chart
A radar (spider) chart lets us compare the full audio fingerprint of each genre simultaneously revealing which dimensions most differentiate genres.

In [ ]:
# Radar chart: mean features per genre
features = ['valence', 'energy', 'danceability', 'acousticness', 'instrumentalness']
genre_means = df.groupby('genre')[features].mean().reset_index()

fig3 = go.Figure()

colors = px.colors.qualitative.Bold
for i, row in genre_means.iterrows():
    vals = [row[f] for f in features] + [row[features[0]]]  # close the loop
    fig3.add_trace(go.Scatterpolar(
        r=vals, theta=features + [features[0]],
        fill='toself', name=row['genre'],
        line_color=colors[i % len(colors)], opacity=0.5
    ))

fig3.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Audio Feature Radar by Genre',
    width=750, height=600,
    paper_bgcolor='white',
    font=dict(family='Arial')
)
fig3.show()


---
## 4. Danceability vs. Tempo
Tempo (BPM) and danceability don't always correlate fast songs aren't always danceable, and slow songs can be surprisingly groovy. This chart examines the relationship by genre.

In [ ]:
fig4 = px.scatter(
    df, x='tempo', y='danceability', color='genre',
    size='energy', size_max=10, opacity=0.6,
    title='Danceability vs. Tempo (bubble size = Energy)',
    labels={'tempo': 'Tempo (BPM)', 'danceability': 'Danceability (0–1)'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    width=900, height=580
)
fig4.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial'))
fig4.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig4.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig4.show()


---
## 5. Acoustic vs. Electronic Spectrum
Acousticness measures how acoustic a track is, while instrumentalness measures the absence of vocals. This spectrum reveals each genre's production identity.

In [ ]:
fig5 = px.scatter(
    df, x='acousticness', y='instrumentalness', color='genre',
    opacity=0.55, trendline='lowess',
    title='Acoustic vs. Instrumental Spectrum by Genre',
    labels={'acousticness': 'Acousticness (Electronic → Acoustic)', 'instrumentalness': 'Instrumentalness (Vocal → Instrumental)'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    width=900, height=580
)
fig5.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial'))
fig5.update_xaxes(showgrid=True, gridcolor='#f0f0f0', range=[0, 1])
fig5.update_yaxes(showgrid=True, gridcolor='#f0f0f0', range=[0, 1])
fig5.show()


---
## 6. Genre Summary Table
Mean values across all key audio features per genre, sorted by valence.

In [ ]:
summary = df.groupby('genre')[['valence', 'energy', 'danceability', 'tempo', 'acousticness', 'instrumentalness', 'loudness']].mean().round(3)
summary = summary.sort_values('valence', ascending=False)
summary.columns = ['Valence', 'Energy', 'Danceability', 'Tempo (BPM)', 'Acousticness', 'Instrumentalness', 'Loudness (dB)']
summary['Tempo (BPM)'] = summary['Tempo (BPM)'].round(1)
summary['Loudness (dB)'] = summary['Loudness (dB)'].round(1)

fig6 = go.Figure(data=[go.Table(
    header=dict(
        values=['Genre'] + list(summary.columns),
        fill_color='#1e1e2e', font=dict(color='white', size=12, family='Arial'),
        align='center', height=35
    ),
    cells=dict(
        values=[summary.index] + [summary[col] for col in summary.columns],
        fill_color=[['#f8f9fa' if i % 2 == 0 else 'white' for i in range(len(summary))]],
        font=dict(size=12, family='Arial'),
        align='center', height=30
    )
)])
fig6.update_layout(title='Genre Audio Feature Summary (Sorted by Valence)', width=950, height=420, paper_bgcolor='white')
fig6.show()


---
## Key Findings

**1. Emotional Quadrant Clustering**  
Metal and Rock dominate the *Dark/Intense* quadrant (low valence, high energy), while Pop and Country skew *Happy/Excited*. Classical and Jazz occupy *Content/Peaceful* territory, calm and not overly negative.

**2. Danceability ≠ Tempo**  
Hip-Hop and R&B achieve the highest danceability scores at *lower* tempos (~95–99 BPM), while Metal runs at 148 BPM with low danceability, confirming tempo alone doesn't drive groove.

**3. Acoustic-Instrumental Axis**  
Classical is simultaneously the most acoustic and most instrumental genre. Electronic music occupies the opposite extreme, synthesized and largely vocal-absent in its purest forms.

**4. Energy Spread**  
Metal (0.92 avg energy) and Electronic (0.81) are clear outliers on the high end. Classical (0.27) is the most distinctly calm genre by a wide margin.

**5. Genre as Emotional Fingerprint**  
Each genre clusters distinctly in the valence-energy space, confirming that genre is a reliable proxy for emotional expectation which has direct implications for mood-based playlist design and recommendation systems.

---
*Dataset: 2,000 synthetic Spotify-style tracks | Analysis: Kaden Van Atta*
